In [1]:
spark

In [2]:
import pandas as pd

users_pd = pd.read_csv("sd254_users.csv")
users_pd["User"] = users_pd.index 
print(users_pd.head(1))

           Person  Current Age  Retirement Age  Birth Year  Birth Month  \
0  Hazel Robinson           53              66        1966           11   

   Gender        Address  Apartment      City State  Zipcode  Latitude  \
0  Female  462 Rose Lane        NaN  La Verne    CA    91750     34.15   

   Longitude Per Capita Income - Zipcode Yearly Income - Person Total Debt  \
0    -117.76                      $29278                 $59696    $127613   

   FICO Score  Num Credit Cards  User  
0         787                 5     0  


In [52]:
dfusers =  spark.createDataFrame(users_pd)

In [7]:
df = spark.read.option("header", True).csv("file:///home/student/Data/Project/credit_card_transactions-ibm_v2.csv")

In [8]:
df.printSchema()

root
 |-- User: string (nullable = true)
 |-- Card: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: string (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- MCC: string (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)



In [9]:
df.select("User").distinct().count()

2000

In [13]:
dfcards = spark.read.option("header", True).csv("file:///home/student/Data/Project/sd254_cards.csv")
dfcards.select("CARD INDEX").count()

6146

In [14]:
df.groupBy("Is Fraud?").count().show()

+---------+--------+
|Is Fraud?|   count|
+---------+--------+
|       No|24357143|
|      Yes|   29757|
+---------+--------+



In [15]:
df.select("Card").count()

24386900

In [17]:
df.groupBy("Card").count().show()

+----+-------+
|Card|  count|
+----+-------+
|   7|  46383|
|   3|2790785|
|   8|   5184|
|   0|8696411|
|   5| 563097|
|   6| 176729|
|   1|6493597|
|   4|1309120|
|   2|4305594|
+----+-------+



In [18]:
dfcards.groupBy("CARD INDEX").count().show()

+----------+-----+
|CARD INDEX|count|
+----------+-----+
|         7|   20|
|         3|  747|
|         8|    3|
|         0| 2000|
|         5|  165|
|         6|   60|
|         1| 1584|
|         4|  371|
|         2| 1196|
+----------+-----+



In [21]:
df.select("Is Fraud?").groupBy("Card").count().show()

AttributeError: 'GroupedData' object has no attribute 'show'

In [23]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in ["Amount", "Merchant State", "Zip", "Errors?", "User", "Card"," Month", "Year", "Day", "Time", "Use Chip", "Merchant Name", "Merchant City", "Is Fraud?"]]).show()v

SyntaxError: invalid syntax (3860751368.py, line 3)

In [24]:
from pyspark.sql.functions import col, sum


cols_to_check = ["Amount", "Merchant State", "Zip", "Errors?", "User", "Card", "Month", "Year", "Day", "Time", "Use Chip", "Merchant Name", "Merchant City", "Is Fraud?"]

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in cols_to_check]).show()

+------+--------------+-------+--------+----+----+-----+----+---+----+--------+-------------+-------------+---------+
|Amount|Merchant State|    Zip| Errors?|User|Card|Month|Year|Day|Time|Use Chip|Merchant Name|Merchant City|Is Fraud?|
+------+--------------+-------+--------+----+----+-----+----+---+----+--------+-------------+-------------+---------+
|     0|       2720821|2878135|23998469|   0|   0|    0|   0|  0|   0|       0|            0|            0|        0|
+------+--------------+-------+--------+----+----+-----+----+---+----+--------+-------------+-------------+---------+



In [25]:
   from pyspark.sql.functions import col, sum, count


fraud_per_card = df.groupBy("User", "Card", "Is Fraud?").count()

fraud_per_card.show()

+----+----+---------+-----+
|User|Card|Is Fraud?|count|
+----+----+---------+-----+
|  59|   0|       No| 2986|
|  74|   1|       No| 5245|
| 148|   3|       No| 1038|
| 215|   0|      Yes|   14|
| 238|   3|       No|   17|
| 279|   0|       No|  115|
| 351|   3|       No|    6|
| 373|   3|       No|   15|
| 404|   0|       No|   36|
| 421|   2|      Yes|    4|
| 455|   0|       No|   25|
| 461|   1|       No| 1197|
| 515|   1|      Yes|   12|
| 525|   1|       No| 3560|
| 615|   4|       No| 1057|
| 649|   0|       No|  218|
| 656|   0|       No|21980|
| 739|   0|      Yes|    4|
| 788|   0|       No| 4822|
| 799|   4|       No| 5988|
+----+----+---------+-----+
only showing top 20 rows



In [26]:


df.filter(col("Is Fraud?") == "Yes").groupBy("Card").count().orderBy("Card").show()

+----+-----+
|Card|count|
+----+-----+
|   0| 8644|
|   1| 7514|
|   2| 5807|
|   3| 4135|
|   4| 2157|
|   5| 1016|
|   6|  350|
|   7|  109|
|   8|   25|
+----+-----+



In [29]:
df.groupBy("Merchant State").count().show()

+--------------+------+
|Merchant State| count|
+--------------+------+
|        Russia|   302|
|      Paraguay|     1|
|            AZ|351114|
|            SC|357973|
|         Yemen|    16|
|       Senegal|    13|
|        Sweden|   496|
|            LA|292575|
|    Cabo Verde|    61|
|            MN|350739|
|      Kiribati|     1|
|        Guyana|    40|
|   Philippines|  1206|
|       Eritrea|    48|
|      Djibouti|    45|
|         Tonga|     2|
|     Singapore|   443|
|      Malaysia|   251|
|            AA|    12|
|            NJ|630317|
+--------------+------+
only showing top 20 rows



In [28]:
from pyspark.sql.functions import col, regexp_replace

df_clean = df.withColumn("Amount", regexp_replace(col("Amount"), "\\$", "").cast("double"))

df_clean.select("Amount").show(5)
df_clean.printSchema()

+------+
|Amount|
+------+
|134.09|
| 38.48|
|120.34|
|128.95|
|104.71|
+------+
only showing top 5 rows

root
 |-- User: string (nullable = true)
 |-- Card: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: string (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- MCC: string (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)



In [32]:
df.filter(col("Is Fraud?") == "Yes").groupBy("Merchant State").count().orderBy(col("count").desc()).show()

+--------------+-----+
|Merchant State|count|
+--------------+-----+
|          null|18349|
|         Italy| 4682|
|            OH|  878|
|            CA|  751|
|       Algeria|  629|
|         Haiti|  375|
|            FL|  314|
|            TX|  305|
|        Mexico|  272|
|        Turkey|  257|
|            NY|  240|
|            NJ|  171|
|            MI|  160|
|            NC|  144|
|            PA|  144|
|       Nigeria|  143|
|            IL|  114|
|            TN|  109|
|            WA|  107|
|            VA|  105|
+--------------+-----+
only showing top 20 rows



In [33]:
from pyspark.sql.functions import avg, max, min, col

df_clean.groupBy("Is Fraud?") \
    .agg(
        avg("Amount").alias("Average_Amount"),
        max("Amount").alias("Max_Amount"),
        min("Amount").alias("Min_Amount")
    ).show()

+---------+------------------+----------+----------+
|Is Fraud?|    Average_Amount|Max_Amount|Min_Amount|
+---------+------------------+----------+----------+
|       No| 43.55464754343359|   12390.5|    -500.0|
|      Yes|108.59087374399306|   5694.44|    -500.0|
+---------+------------------+----------+----------+



In [34]:
df_clean.filter(col("Is Fraud?") == "Yes") \
    .groupBy("MCC") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+----+-----+
| MCC|count|
+----+-----+
|5311| 4824|
|5300| 2201|
|5310| 2152|
|4829| 1607|
|5912| 1057|
|5411|  943|
|5815|  879|
|5651|  849|
|5732|  843|
|5719|  717|
|5094|  567|
|4121|  558|
|5211|  475|
|5045|  470|
|5814|  466|
|5712|  418|
|4722|  406|
|4814|  401|
|5541|  354|
|7922|  328|
+----+-----+
only showing top 20 rows



In [35]:
from pyspark.sql.functions import sum

df_clean.filter(col("Is Fraud?") == "Yes") \
    .groupBy("Use Chip") \
    .agg(sum("Amount").alias("Total_Stolen")) \
    .orderBy(col("Total_Stolen").desc()) \
    .show()

+------------------+-----------------+
|          Use Chip|     Total_Stolen|
+------------------+-----------------+
|Online Transaction|2165035.710000001|
| Swipe Transaction|656792.7699999998|
|  Chip Transaction|409510.1499999999|
+------------------+-----------------+



In [37]:
from pyspark.sql.functions import sum

df_clean.filter(col("Is Fraud?") == "Yes") \
    .groupBy("Use Chip") \
    .count() \
.show()

+------------------+-----+
|          Use Chip|count|
+------------------+-----+
| Swipe Transaction| 6572|
|  Chip Transaction| 4836|
|Online Transaction|18349|
+------------------+-----+



In [38]:
from pyspark.sql.functions import col, concat_ws, to_timestamp

df_clean = df_clean.withColumn(
    "datetime_str", 
    concat_ws(" ", concat_ws("-", col("Year"), col("Month"), col("Day")), col("Time"))
)

df_clean = df_clean.withColumn(
    "timestamp", 
    to_timestamp(col("datetime_str"), "yyyy-M-d H:m")
)

df_clean.select("Year", "Month", "Day", "Time", "datetime_str", "timestamp").show( truncate=False)

df_clean.printSchema()

+----+-----+---+-----+--------------+-------------------+
|Year|Month|Day|Time |datetime_str  |timestamp          |
+----+-----+---+-----+--------------+-------------------+
|2002|9    |1  |06:21|2002-9-1 06:21|2002-09-01 06:21:00|
|2002|9    |1  |06:42|2002-9-1 06:42|2002-09-01 06:42:00|
|2002|9    |2  |06:22|2002-9-2 06:22|2002-09-02 06:22:00|
|2002|9    |2  |17:45|2002-9-2 17:45|2002-09-02 17:45:00|
|2002|9    |3  |06:23|2002-9-3 06:23|2002-09-03 06:23:00|
|2002|9    |3  |13:53|2002-9-3 13:53|2002-09-03 13:53:00|
|2002|9    |4  |05:51|2002-9-4 05:51|2002-09-04 05:51:00|
|2002|9    |4  |06:09|2002-9-4 06:09|2002-09-04 06:09:00|
|2002|9    |5  |06:14|2002-9-5 06:14|2002-09-05 06:14:00|
|2002|9    |5  |09:35|2002-9-5 09:35|2002-09-05 09:35:00|
|2002|9    |5  |20:18|2002-9-5 20:18|2002-09-05 20:18:00|
|2002|9    |5  |20:41|2002-9-5 20:41|2002-09-05 20:41:00|
|2002|9    |6  |06:16|2002-9-6 06:16|2002-09-06 06:16:00|
|2002|9    |7  |06:16|2002-9-7 06:16|2002-09-07 06:16:00|
|2002|9    |7 

In [39]:
df_clean = df_clean.drop("Year", "Month", "Day", "Time", "datetime_str")

df_clean.printSchema()

root
 |-- User: string (nullable = true)
 |-- Card: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: string (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- MCC: string (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [40]:
fill_values = {
    "Merchant State": "ONLINE",
    "Zip": "00000",
    "Errors?": "No Error"
}

df_clean = df_clean.fillna(fill_values)

df_clean.filter(col("Merchant State") == "ONLINE").show()

+----+----+------+------------------+--------------------+-------------+--------------+-----+----+--------------------+---------+-------------------+
|User|Card|Amount|          Use Chip|       Merchant Name|Merchant City|Merchant State|  Zip| MCC|             Errors?|Is Fraud?|          timestamp|
+----+----+------+------------------+--------------------+-------------+--------------+-----+----+--------------------+---------+-------------------+
|   0|   0| 53.91|Online Transaction|-9092677072201095172|       ONLINE|        ONLINE|00000|4900|            No Error|       No|2002-09-05 20:41:00|
|   0|   0| 144.9|Online Transaction|-8338381919281017248|       ONLINE|        ONLINE|00000|4899|            No Error|       No|2002-09-09 20:02:00|
|   0|   0|127.32|Online Transaction|-7421093378627544099|       ONLINE|        ONLINE|00000|5311|            No Error|       No|2002-09-30 06:21:00|
|   0|   0|139.39|Online Transaction|-7421093378627544099|       ONLINE|        ONLINE|00000|5311|  

In [45]:
from pyspark.sql.functions import col

df_clean.groupBy("Merchant State").count().orderBy(col("count").desc()).show(280, truncate=False)

+--------------------------------+-------+
|Merchant State                  |count  |
+--------------------------------+-------+
|ONLINE                          |2720821|
|CA                              |2591830|
|TX                              |1793298|
|FL                              |1458699|
|NY                              |1446864|
|OH                              |895970 |
|IL                              |850074 |
|PA                              |839647 |
|NC                              |779234 |
|GA                              |638884 |
|NJ                              |630317 |
|MI                              |618407 |
|IN                              |613432 |
|WA                              |537762 |
|TN                              |504116 |
|VA                              |425216 |
|MD                              |377134 |
|MA                              |364359 |
|SC                              |357973 |
|AZ                              |351114 |
|MN        

In [46]:
from pyspark.sql import Row
from pyspark.sql.functions import col

locations_data = {
    "ONLINE": (0.0, 0.0),
    "CA": (36.77, -119.41), "TX": (31.96, -99.90), "FL": (27.99, -81.76),
    "NY": (40.71, -74.00), "OH": (40.41, -82.90), "IL": (40.63, -89.39),
    "PA": (41.20, -77.19), "NC": (35.75, -79.01), "GA": (32.16, -82.90),
    "NJ": (40.05, -74.40), "MI": (44.31, -85.60), "IN": (39.84, -86.21),
    "WA": (47.75, -120.74), "TN": (35.51, -86.58), "VA": (37.43, -78.65),
    "MD": (39.04, -76.64), "MA": (42.40, -71.38), "SC": (33.83, -81.16),
    "AZ": (34.04, -111.09), "MN": (46.72, -94.68), "WI": (43.78, -88.78),
    "AL": (32.31, -86.90), "KY": (37.83, -84.27), "OK": (35.00, -97.09),
    "MO": (37.96, -91.83), "LA": (31.16, -91.86), "CO": (39.55, -105.78),
    "IA": (41.87, -93.09), "AR": (35.20, -92.83), "CT": (41.60, -72.73),
    "MS": (32.35, -89.39), "KS": (39.01, -98.48), "OR": (43.80, -120.55),
    "NM": (34.51, -105.87), "NV": (38.80, -116.41), "UT": (39.32, -111.09),
    "HI": (19.89, -155.66), "WV": (38.59, -80.45), "NH": (43.19, -71.57),
    "RI": (41.58, -71.47), "NE": (41.12, -98.26), "ID": (44.06, -114.74),
    "DE": (39.31, -75.50), "ME": (45.25, -69.44), "SD": (43.96, -99.90),
    "ND": (47.55, -101.00), "MT": (46.96, -110.45), "VT": (44.55, -72.57),
    "DC": (38.90, -77.03), "WY": (43.07, -107.29), "AK": (64.20, -152.49),
    "Mexico": (23.63, -102.55), "Canada": (56.13, -106.34),
    "Italy": (41.87, 12.56), "United Kingdom": (55.37, -3.43),
    "France": (46.22, 2.21), "Germany": (51.16, 10.45),
    "China": (35.86, 104.19), "Japan": (36.20, 138.25),
    "India": (20.59, 78.96), "Spain": (40.46, -3.74),
    "Dominican Republic": (18.73, -70.16), "Netherlands": (52.13, 5.29),
    "Jamaica": (18.10, -77.29), "South Korea": (35.90, 127.76),
    "Switzerland": (46.81, 8.22), "Israel": (31.04, 34.85),
    "Ireland": (53.14, -7.69), "The Bahamas": (25.03, -77.39),
    "Taiwan": (23.69, 120.96), "Colombia": (4.57, -74.29),
    "Austria": (47.51, 14.55), "Costa Rica": (9.74, -83.75),
    "Thailand": (15.87, 100.99), "Philippines": (12.87, 121.77),
    "Brazil": (-14.23, -51.92), "Hong Kong": (22.31, 114.16),
    "Australia": (-25.27, 133.77), "Aruba": (12.52, -69.96),
    "Greece": (39.07, 21.82), "Peru": (-9.18, -75.01),
    "Norway": (60.47, 8.46), "Czech Republic": (49.81, 15.47),
    "Algeria": (28.03, 1.65), "Pakistan": (30.37, 69.34),
    "Denmark": (56.26, 9.50), "Portugal": (39.39, -8.22),
    "United Arab Emirates": (23.42, 53.84), "South Africa": (-30.55, 22.93),
    "Sweden": (60.12, 18.64), "Belgium": (50.50, 4.46),
    "Turkey": (38.96, 35.24), "Poland": (51.91, 19.14),
    "Haiti": (18.97, -72.28), "Singapore": (1.35, 103.81),
    "Hungary": (47.16, 19.50), "New Zealand": (-40.90, 174.88),
    "Finland": (61.92, 25.74), "Russia": (61.52, 105.31),
    "Indonesia": (-0.78, 113.92), "Egypt": (26.82, 30.80),
    "Saudi Arabia": (23.88, 45.07), "Argentina": (-38.41, -63.61),
    "Malaysia": (4.21, 101.97), "Nigeria": (9.08, 8.67),
    "Vatican City": (41.90, 12.45), "Macedonia": (41.60, 21.74),
    "Barbados": (13.19, -59.54), "Vietnam": (14.05, 108.27),
    "Kenya": (-0.02, 37.90), "Iceland": (64.96, -19.02),
    "Chile": (-35.67, -71.54), "Slovenia": (46.15, 14.99),
    "Luxembourg": (49.81, 6.12), "Croatia": (45.10, 15.20),
    "Serbia": (44.01, 21.00), "Bangladesh": (23.68, 90.35),
    "Ukraine": (48.37, 31.16), "Oman": (21.51, 55.92),
    "Sri Lanka": (7.87, 80.77), "Slovakia": (48.66, 19.69),
    "Mongolia": (46.86, 103.84), "Ecuador": (-1.83, -78.18),
    "Honduras": (15.20, -86.24), "Uganda": (1.37, 32.29),
    "Romania": (45.94, 24.96), "Latvia": (56.87, 24.60),
    "Uruguay": (-32.52, -55.76), "Estonia": (58.59, 25.01),
    "Morocco": (31.79, -7.09), "Ghana": (7.94, -1.02),
    "Venezuela": (6.42, -66.58), "Azerbaijan": (40.14, 47.57),
    "Bulgaria": (42.73, 25.48), "Tanzania": (-6.36, 34.88),
    "Panama": (8.53, -80.78)
}

rows = [Row(State=k, lat_map=v[0], lon_map=v[1]) for k, v in locations_data.items()]
geo_df = spark.createDataFrame(rows)

df_clean = df_clean.join(geo_df, df_clean["Merchant State"] == geo_df["State"], "left")

df_clean = df_clean.fillna({"lat_map": -1.0, "lon_map": -1.0})

df_clean = df_clean.withColumnRenamed("lat_map", "lat") \
                   .withColumnRenamed("lon_map", "lon")

df_clean = df_clean.drop("State")

df_clean.select("Merchant State", "lat", "lon").show().

+--------------+-----+------+
|Merchant State|  lat|   lon|
+--------------+-----+------+
|      Paraguay| -1.0|  -1.0|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
|        Russia|61.52|105.31|
+--------------+-----+------+
only showing top 20 rows



In [50]:
df_clean.select("Merchant State", "lat", "lon").distinct().show()

+--------------+-----+-------+
|Merchant State|  lat|    lon|
+--------------+-----+-------+
|      Paraguay| -1.0|   -1.0|
|        Russia|61.52| 105.31|
|            AZ|34.04|-111.09|
|            SC|33.83| -81.16|
|         Yemen| -1.0|   -1.0|
|       Senegal| -1.0|   -1.0|
|    Cabo Verde| -1.0|   -1.0|
|            LA|31.16| -91.86|
|        Sweden|60.12|  18.64|
|      Kiribati| -1.0|   -1.0|
|            MN|46.72| -94.68|
|        Guyana| -1.0|   -1.0|
|       Eritrea| -1.0|   -1.0|
|   Philippines|12.87| 121.77|
|      Djibouti| -1.0|   -1.0|
|         Tonga| -1.0|   -1.0|
|            AA| -1.0|   -1.0|
|      Malaysia| 4.21| 101.97|
|     Singapore| 1.35| 103.81|
|          Fiji| -1.0|   -1.0|
+--------------+-----+-------+
only showing top 20 rows



In [62]:
users_df = spark.read.option("header",True).csv("file:///home/student/Data/Project/sd254_users.csv")
Credit_card_holders = users_df.join(dfcards, dfcards.User == users_df.User,"full")

AttributeError: 'DataFrame' object has no attribute 'User'

In [2]:
df.show()

NameError: name 'df' is not defined

In [61]:

del dfusers

In [71]:
dfuser = spark.createDataFrame(users_pd)

In [ ]:
dfuser = spark.read.option("header",True).csv("file:///home/student/Data/Project/sd254_users.csv")
Credit_card_holders = dfuser.join(dfcards, dfcards.User == dfuser.User,"full")